# mp4 文字起こし + 前処理 (Colab / GPU)

パイプライン: `mp4` → 音声抽出 → **demucs** ボーカル分離 → 16k mono → **whisperX** (large-v3, 日本語, 単語timestamp + 話者分離) → 話者ラベル付き `txt`

## 事前準備
1. ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択
2. ローカルの `interfacex/chore/static` を Google Drive にアップロード（mp4 を置く）
3. 話者分離(diarization)を使う場合、HuggingFace token が必要:
   - https://huggingface.co/settings/tokens でトークン作成（Read 権限でOK）
   - 以下モデルの利用規約に同意（要ログイン、"Agree and access repository" 押す）:
     - https://huggingface.co/pyannote/speaker-diarization-community-1
   - ※ whisperx の旧バージョンは `pyannote/speaker-diarization-3.1` + `pyannote/segmentation-3.0` を使う。エラーメッセージに出たモデル名のページで同意すること
   - 左サイドバーの鍵アイコン(Secrets) → 名前 `HF_TOKEN` でトークンを登録 → 「ノートブックからのアクセス」をON（マウントセルが `userdata.get('HF_TOKEN')` で読む）

## 1. GPU確認

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'GPU無効: ランタイムのタイプをGPUに変更'
print('CUDA OK:', torch.cuda.get_device_name(0))

## 2. 依存インストール (数分かかる)

In [ ]:
# Colab同梱の torch/torchvision/transformers と whisperx 3.8+ が衝突すると
# "cannot import name 'Pipeline' from 'transformers'" になる (torchvision 不整合が多い)
!pip -q install -U pip
!pip -q install -U "whisperx>=3.8" demucs "transformers>=4.48,<5" "torchvision~=0.23.0" "torchcodec>=0.6.0,<0.8.0"
!apt -qq install -y ffmpeg >/dev/null

# 再起動前に import 確認 (ここで失敗したら上の pip 出力を確認して再実行)
import importlib
import transformers
print('transformers:', transformers.__version__)
importlib.import_module('transformers.pipelines')
print('Pipeline OK')

print('インストール完了 -> 自動再起動します。再起動後はこのセルを飛ばしてマウントセル(#3)から実行')
import os; os.kill(os.getpid(), 9)

## 3. Drive マウント + パス設定

In [ ]:
from datetime import datetime
from google.colab import drive, userdata

#@markdown Driveを再マウント (mp4が古い表示のとき True)
FORCE_REMOUNT = True  #@param {type:'boolean'}
drive.mount('/content/drive', force_remount=FORCE_REMOUNT)

import os

#@markdown Drive内の static フォルダのパス (mp4が入っている場所)
STATIC_DIR = '/content/drive/MyDrive/chore/static'  #@param {type:'string'}
#@markdown 文字起こしエンジン言語
LANGUAGE = 'ja'  #@param {type:'string'}
#@markdown whisperモデル
MODEL = 'large-v3'  #@param ['large-v3','large-v2','medium','small']
#@markdown 話者分離(diarization)を有効化
USE_DIARIZATION = True  #@param {type:'boolean'}
#@markdown TODO: HuggingFace tokenをcolab secretに設定
HF_TOKEN = userdata.get('HF_TOKEN')

VIDEO_EXTS = {'.mp4', '.m4v', '.mov', '.mkv', '.webm'}

def find_videos(root):
    """static 以下を再帰検索 (transcripts/ は除外). 拡張子は大文字小文字無視."""
    videos = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d != 'transcripts']
        for fn in filenames:
            if os.path.splitext(fn)[1].lower() in VIDEO_EXTS:
                videos.append(os.path.join(dirpath, fn))
    return sorted(videos)

TRANSCRIPTS_DIR = os.path.join(STATIC_DIR, 'transcripts')
PREPROCESSED_DIR = os.path.join(TRANSCRIPTS_DIR, 'preprocessed')
os.makedirs(TRANSCRIPTS_DIR, exist_ok=True)
os.makedirs(PREPROCESSED_DIR, exist_ok=True)

mp4s = find_videos(STATIC_DIR)
assert mp4s, f'動画が見つからない: {STATIC_DIR}'

# 診断: 直下だけだと何本見えるか
import glob
top_level = sorted(glob.glob(os.path.join(STATIC_DIR, '*.mp4')))
if len(top_level) < len(mp4s):
    print(f'※ 直下 *.mp4 は {len(top_level)} 本 / 再帰検索で {len(mp4s)} 本')

print(f'{len(mp4s)}本の動画:')
for p in mp4s:
    rel = os.path.relpath(p, STATIC_DIR)
    mtime = datetime.fromtimestamp(os.path.getmtime(p)).strftime('%Y-%m-%d %H:%M')
    print(f'  {rel}  ({mtime})')

## 4. 前処理関数 (mp4 → demucs vocals → 16k mono)

出力構造（既存ローカルと同形）:
```
transcripts/preprocessed/<name>/source.wav
transcripts/preprocessed/<name>/vocals_16k_mono.wav
transcripts/preprocessed/<name>/demucs/htdemucs/source/{vocals,no_vocals}.wav
```

In [ ]:
import subprocess, shutil

def run(cmd):
    subprocess.run(cmd, check=True)

def preprocess(mp4_path):
    name = os.path.splitext(os.path.basename(mp4_path))[0]
    outdir = os.path.join(PREPROCESSED_DIR, name)
    os.makedirs(outdir, exist_ok=True)
    source_wav = os.path.join(outdir, 'source.wav')
    vocals_16k = os.path.join(outdir, 'vocals_16k_mono.wav')

    if os.path.exists(vocals_16k):
        print(f'[skip] {name} 前処理済')
        return vocals_16k

    # mp4 -> source.wav (44.1k stereo)
    run(['ffmpeg','-y','-i',mp4_path,'-vn','-ac','2','-ar','44100',source_wav])

    # demucs ボーカル分離 (htdemucs, vocals/no_vocals)
    demucs_out = os.path.join(outdir, 'demucs')
    run(['python','-m','demucs','--two-stems=vocals','-n','htdemucs','-o',demucs_out,source_wav])
    vocals = os.path.join(demucs_out,'htdemucs','source','vocals.wav')

    # vocals -> 16k mono + ダイナミックレンジ正規化 (whisper入力)
    # dynaudnorm: 小音量の声を底上げ→テレビに埋もれた小声をVADが拾えるよう底上げ
    #   f=フレーム長(ms), g=ガウシアン窓, p=ターゲットピーク, m=最大ゲイン
    run(['ffmpeg','-y','-i',vocals,'-ac','1','-ar','16000',
         '-af','dynaudnorm=f=150:g=15:p=0.9:m=20',vocals_16k])
    print(f'[ok] {name} 前処理完了')
    return vocals_16k

## 5. 出力ライター (txt のみ・話者ラベル付き)

In [ ]:
def _spk(seg):
    sp = seg.get('speaker')
    return f'[{sp}] ' if sp else ''

def write_outputs(result, name):
    segs = result['segments']
    base = os.path.join(TRANSCRIPTS_DIR, name)

    with open(base+'.txt','w',encoding='utf-8') as f:
        for s in segs:
            f.write(_spk(s) + s['text'].strip() + '\n')
    print(f'[ok] {name} 出力 -> {base}.txt')

## 6. モデルロード (1回だけ)

In [ ]:
# 依存未インストール / 再起動漏れの早期検出
import importlib
import transformers
try:
    from transformers import Pipeline  # noqa: F401
except ImportError as e:
    raise RuntimeError(
        'transformers の import に失敗しました。\n'
        f'  現在の transformers: {transformers.__version__}\n'
        '  対処: セル#2(依存インストール)を実行 → 自動再起動 → セル#3から再実行'
    ) from e
importlib.import_module('transformers.pipelines')

import whisperx, gc
device = 'cuda'
compute_type = 'float16'

# 小声(テレビ裏の聞き取りづらい声)を捨てないよう感度を上げる
#   no_speech_threshold↑ : 無音判定を厳しく→小声セグメントを残す
#   log_prob_threshold↓  : 低確信デコードも棄却しない
#   compression_ratio_threshold↑ : 繰返し気味でも棄却しない
ASR_OPTIONS = {
    'no_speech_threshold': 0.8,       # default 0.6
    'log_prob_threshold': -2.0,       # default -1.0
    'compression_ratio_threshold': 3.0,  # default 2.4
    'condition_on_previous_text': False, # テレビ混在での文脈ドリフト/幻聴抑制
}
#   vad_onset/offset↓ : 弱い音声の開始/継続を取りこぼさない
VAD_OPTIONS = {
    'vad_onset': 0.3,    # default 0.500
    'vad_offset': 0.2,   # default 0.363
}

asr_model = whisperx.load_model(
    MODEL, device, compute_type=compute_type, language=LANGUAGE,
    asr_options=ASR_OPTIONS, vad_options=VAD_OPTIONS,
)
align_model, align_meta = whisperx.load_align_model(language_code=LANGUAGE, device=device)

diarize_model = None
if USE_DIARIZATION:
    assert HF_TOKEN, 'HF_TOKEN未設定: 設定セルで入力するか USE_DIARIZATION=False'
    try:
        from whisperx.diarize import DiarizationPipeline
    except ImportError:
        from whisperx import DiarizationPipeline
    # whisperx バージョンで引数名が use_auth_token / token と異なる
    try:
        diarize_model = DiarizationPipeline(use_auth_token=HF_TOKEN, device=device)
    except TypeError:
        diarize_model = DiarizationPipeline(token=HF_TOKEN, device=device)
print('models loaded')

## 7. 一括実行

In [ ]:
BATCH_SIZE = 16

for mp4 in mp4s:
    name = os.path.splitext(os.path.basename(mp4))[0]
    print(f'\n=== {name} ===')
    audio_path = preprocess(mp4)

    audio = whisperx.load_audio(audio_path)
    result = asr_model.transcribe(audio, batch_size=BATCH_SIZE, language=LANGUAGE)
    result = whisperx.align(result['segments'], align_model, align_meta, audio, device,
                            return_char_alignments=False)

    if diarize_model is not None:
        diarize_segments = diarize_model(audio)
        result = whisperx.assign_word_speakers(diarize_segments, result)

    result['language'] = LANGUAGE
    write_outputs(result, name)
    gc.collect(); torch.cuda.empty_cache()

print('\n全件完了')